# Module 7: Datatypes in Polars

In [ ]:
import polars as pl

In [ ]:
zone_column_rename_mapping = {
    "LocationID": "location_id",
    "Borough": "borough",
    "Zone": "zone",
}

zones_df = pl.read_parquet("./../data/taxi_zone_lookup.parquet").rename(
    zone_column_rename_mapping
)

In [ ]:
yellow_rides_column_rename_mapping = {
    "VendorID": "vendor_id",
    "RatecodeID": "rate_code_id",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Airport_fee": "airport_fee",
}

zones_df_columns = ["borough", "zone", "service_zone"]

rides_df = (
    pl.read_parquet("./../data/yellow_tripdata_2024-03.parquet")
    .rename(yellow_rides_column_rename_mapping)
    .join(
        zones_df,
        left_on="pickup_location_id",
        right_on="location_id",
    )
    .rename({zone: f"pu_{zone}" for zone in zones_df_columns})
    .join(
        zones_df,
        left_on="dropoff_location_id",
        right_on="location_id",
    )
    .rename({zone: f"do_{zone}" for zone in zones_df_columns})
)

In [ ]:
rides_df

In [ ]:
rides_df.schema

In [ ]:
rides_df.select(
    "tpep_pickup_datetime", "total_amount", "airport_fee", "pu_zone"
).with_columns(
    pl.col("pu_zone").str.contains("Airport").alias("is_airport_pickup")
).filter("is_airport_pickup")

In [ ]:
rides_df.select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone").str.split(by=" ").name.suffix("_splitted"),
)

In [ ]:
rides_df.select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone").str.len_chars().name.suffix("_length"),
)

In [ ]:
rides_df.select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone").str.to_uppercase().name.suffix("_uppercase"),
    pl.col("pu_service_zone").str.to_lowercase().name.suffix("_lowercase"),
    pl.col("pu_service_zone")
    .str.to_uppercase()
    .str.replace(" ", "_")
    .name.suffix("_uppercase_wo_space"),
)

## .list method

In [ ]:
rides_df.select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone").str.split(" ").name.suffix("_splitted"),
)

In [ ]:
rides_df.group_by(
    pl.col("pu_service_zone")
    .str.split(by=" ")
    .list.len()
    .alias("pu_service_zone_num_words")
).agg(pl.len())

In [ ]:
rides_df.with_columns(
    pl.col("pu_service_zone").str.split(by=" ").name.suffix("_splitted")
).select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone_splitted").list.reverse().name.suffix("_reversed"),
)

In [ ]:
rides_df.with_columns(
    pl.col("pu_service_zone").str.split(by=" ").name.suffix("_splitted")
).select(
    "tpep_pickup_datetime",
    "pu_service_zone",
    pl.col("pu_service_zone_splitted").list.get(0).name.suffix("_first"),
)

In [ ]:
zones_df

In [ ]:
zones_df.with_columns(
    pl.col("zone").str.split(by=" ").alias("zone_splitted")
).with_columns(
    pl.col("zone_splitted").list.get(1, null_on_oob=True).alias("second_element")
).group_by("second_element").agg(pl.len()).sort("len", descending=True)

In [ ]:
zones_df.select(pl.col("zone").str.contains("North").alias("contains_north")).sum()

In [ ]:
rides_df

In [ ]:
rides_df.select(
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_days()
    .alias("number_of_days")
).filter(pl.col("number_of_days").ge(1))

In [ ]:
rides_df.select(
    "do_zone",
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds()
    .alias("total_seconds_duration"),
).sort("total_seconds_duration", descending=True)

In [ ]:
rides_df.with_columns(
    pl.col("tpep_pickup_datetime").dt.weekday().alias("tpep_weekday"),
    pl.col("tpep_pickup_datetime").dt.hour().alias("tpep_hour"),
).group_by(["tpep_weekday", "tpep_hour"]).agg(
    pl.col("fare_amount").mean().alias("avg_fare_amount")
).sort("avg_fare_amount", descending=True)

In [ ]:
zones_df.with_columns(
    pl.col("zone").str.split(" ").list.get(0).str.reverse().alias("reverse")
).group_by("reverse").agg(pl.len()).sort("len", descending=True)